# Four research groups, one exchange

Same walk-through as `docs/examples/four-research-groups/run_exchange.py`.
Four groups describe one Swiss electricity system. Each keeps its own files.
Every hand-over goes through a CESDM YAML file.

| Group | Files | What differs |
|---|---|---|
| A | semicolon CSV, German headers | technology is a number (`6` = reservoir hydro); reservoir is a second table |
| B | Excel workbook | technology is a fuel name (`hydro`, `gas`); energy is in GWh |
| C | one nested YAML file | capacities in GW and TWh; region code `CHE`; reservoir sits on the plant |
| D | flat JSON | capacities in kW; efficiency in percent; reservoir sits on the plant |

Each group writes one adapter, `to_cesdm` and `from_cesdm`. No group reads another group's format.

The system is 58 TWh/year of demand, nuclear 3.0 GW, reservoir hydro 8.5 GW with 8.8 TWh of storage and 20 TWh/year of inflow, run-of-river 4.0 GW with 16 TWh/year of inflow, photovoltaics 6.5 GW, wind 0.1 GW, and a 0.4 GW gas plant.

Group B's adapter reads Excel, so the kernel needs `openpyxl` (`pip install -e ".[excel]"` in that same Python). Start Jupyter with that interpreter (`python -m jupyter lab`). Another `jupyter` on the PATH, for example Homebrew, is a different environment and raises `No module named 'openpyxl'`.


## Setup

The next cell finds the repository (the folder that contains `schemas/cesdm`), imports the four adapters through `run_exchange.py`, and writes results under `output/four-research-groups/`. That folder is generated; it is not part of the repository.


In [1]:
from pathlib import Path
import os
import sys

here = Path.cwd().resolve()
REPO = next(
    (p for p in (here, *here.parents) if (p / "schemas" / "cesdm").is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError(
        "Could not find schemas/cesdm. Open this notebook from the cesdm-toolbox checkout."
    )

os.chdir(REPO)
groups = REPO / "docs" / "examples" / "four-research-groups"
if str(groups) not in sys.path:
    sys.path.insert(0, str(groups))

from compare import compare
from run_exchange import GROUPS, OUT, read_yaml, receive, to_cesdm

OUT.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO)
print("Output:    ", OUT.relative_to(REPO))


Repository: /Users/demirayt/Downloads/cesdm-toolbox
Output:     output/four-research-groups


## 1. Each group writes its system as CESDM

`to_cesdm` opens an empty model, reads only that group's files, and writes entities, relations and attributes. Codes, units and the place where the reservoir lives are resolved here, inside the adapter.

`validate()` checks the result against the schema. An empty list means the file is valid CESDM and can be sent. The YAML files are what the other groups will load. They do not load the CSV, the workbook, or the JSON.


In [2]:
for group, (_, native, _, _, _) in GROUPS.items():
    model = to_cesdm(group, native)
    errors = model.validate()
    assert not errors, errors
    model.export_yaml(OUT / f"group_{group.lower()}.yaml")
    print(f"Group {group}  {native.name:28s} → group_{group.lower()}.yaml   valid")


Group A  group-a-csv                  → group_a.yaml   valid


Group B  switzerland_dispatch.xlsx    → group_b.yaml   valid


Group C  system.yaml                  → group_c.yaml   valid


Group D  network.json                 → group_d.yaml   valid


## 2. Do the four files describe the same system?

The plant identifiers do not match: Group A says `CH_NUC`, Group C says `nuc`, Group D says `che-nuc`. Comparison therefore ignores identifiers. `compare` matches a plant on its region and its technology category, then checks capacity, efficiency, variable cost, storage and inflow in CESDM units.

A blank efficiency on wind and photovoltaics matches a blank efficiency. It is not treated as a difference.


In [3]:
original = read_yaml("A", OUT / "group_a.yaml")
for group in "BCD":
    other = read_yaml(group, OUT / f"group_{group.lower()}.yaml")
    diffs = compare(original, other, "A", group)
    print(f"A vs {group}: {len(diffs)} differences")
    for line in diffs:
        print("   ", line)


A vs B: 0 differences


A vs C: 0 differences


A vs D: 0 differences


## 3. Group C receives Group A's system

Group C does not open `plants.csv`. It loads `group_a.yaml` and its own adapter writes C's representation: GW and TWh, the reservoir nested on the plant.

The system information comes from the sender; the representation comes from the receiver. A's numeric technology code and its separate storage table are gone. The demand, the 8.5 GW and the 8.8 TWh are not.


In [4]:
received_by_c = receive("C", read_yaml("A", OUT / "group_a.yaml"))
print("wrote", received_by_c.relative_to(REPO))

lines = received_by_c.read_text(encoding="utf-8").splitlines()
hydro = next(i for i, line in enumerate(lines) if "reservoir_hydro" in line)
for line in [l for l in lines if "demand_twh" in l] + lines[hydro - 1:hydro + 7]:
    print(" ", line)


wrote output/four-research-groups/received-by-c/system.yaml
      demand_twh: 58
      - id: CH_HYD_RES
        tech: reservoir_hydro
        cap_gw: 8.5
        var_cost: 0
        eta: 0.9
        reservoir_id: CH_RES
        reservoir_twh: 8.8
        inflow_twh: 20


## 4. Group A receives Group C's system

The other direction. A's adapter writes semicolon-separated CSV again and puts technologies back into its own codes. Storage and inflow go back into `storage.csv`.

The identifiers in those files are the sender's. This CESDM file came from Group C, so the rows are named `nuc` and `hydro`, and the reservoir id is C's `CHE_RES`. Codes, units and the split between `plants.csv` and `storage.csv` are A's.


In [5]:
received_by_a = receive("A", read_yaml("C", OUT / "group_c.yaml"))
print("wrote", received_by_a.relative_to(REPO))
for name in ("plants.csv", "storage.csv"):
    print(name)
    for line in (received_by_a / name).read_text(encoding="utf-8").splitlines():
        print(" ", line)


wrote output/four-research-groups/received-by-a/csv
plants.csv
  ID;Region;Pmax;Typ;Eta;VarKosten
  ccgt;CH;400;1;58;45
  nuc;CH;3000;2;33;9
  wind;CH;100;3;;0
  pv;CH;6500;4;;0
  hydro;CH;8500;6;90;0
  ror;CH;4000;7;90;0
storage.csv
  ID;Region;E_speicher;Zufluss
  CHE_RES;CH;8800000;20000000
  ror_inflow;CH;0;16000000


## 5. Round trip A → C → A

Send A's system to C, then straight back to A, and compare the result with the CESDM model from step 1.

Zero differences means this pair of adapters kept the system: capacities, efficiencies, costs, storage and inflow. Identifiers may change on the way through C. The comparison does not use them.


In [6]:
back = to_cesdm(
    "A",
    receive(
        "A",
        to_cesdm("C", receive("C", read_yaml("A", OUT / "group_a.yaml"))),
    ),
)
diffs = compare(original, back, "before", "after")
print(f"{len(diffs)} differences")
for line in diffs:
    print(" ", line)


0 differences


## 6. Every group with every other group

Twelve directed exchanges: each of A, B, C and D sends its CESDM file to each of the other three. The receiver writes its own files, reads them back through its adapter, and the result is compared with what was sent.

Twelve exchanges, four adapters. A fifth group would add one more adapter and could exchange with these four on the same day. The existing adapters stay as they are.


In [7]:
preserved = 0
for sender in GROUPS:
    sent = read_yaml(sender, OUT / f"group_{sender.lower()}.yaml")
    for receiver in GROUPS:
        if receiver == sender:
            continue
        arrived = to_cesdm(receiver, receive(receiver, sent))
        diffs = compare(sent, arrived, sender, receiver)
        preserved += not diffs
        print(f"{sender} → {receiver}: {len(diffs)} differences")

print(f"{preserved} of 12 exchanges preserve the system")


A → B: 0 differences


A → C: 0 differences


A → D: 0 differences


B → A: 0 differences


B → C: 0 differences


...

C → B: 0 differences


C → D: 0 differences


D → A: 0 differences


D → B: 0 differences


D → C: 0 differences
12 of 12 exchanges preserve the system
